# Notebook 06 of 7 — Backtest + Validation

*Portfolio Intelligence Engine — User Guide Series.*
[Series README](README.md) · [Story Bible](STORY_BIBLE.md) · Filed under
epic [#1352](https://github.com/prajoria/OpenBB/issues/1352).

---

## Where we are in Sam's story

NB05's what-if said Sam's intuitive trades would hurt. NB05's revised plan was to rotate toward high owner-earnings names inside the basket. Sam's about to paper-trade that plan monthly for a quarter and then decide whether to run it live. Except — is the underlying strategy any good, or does it just happen to look good this month?

By the end of this notebook we will be able to answer one question:

> *Does 'rebalance monthly to the top-5 owner-earnings yield inside my basket' have real edge, or did I get lucky?*


In [ ]:
# [CODE PLACEHOLDER — Phase B] environment sanity — assert .venv_portfolio is active; STATE dir created; friendly halt with setup command if not


## 1. From rationale to `BacktestConfig`

`openbb_backtest` takes a config object — universe, entry rule, exit
rule, rebalance cadence, benchmark. Translating Sam's NB05 rationale:

- **Universe:** the through-line basket (10 names + ETFs)
- **Entry rule:** monthly, top-5 by trailing-12-month owner-earnings yield
- **Exit rule:** on rebalance if no longer top-5, or on trailing-stop
- **Benchmark:** SPY
- **Lookback:** 5 years

*The code cell below encodes this as a `BacktestConfig`.*

In [ ]:
# [CODE PLACEHOLDER — Phase B] build BacktestConfig from NB05 rationale (universe = basket, entry = monthly top-5 owner-earnings yield, exit = rebalance/stop, bench = SPY)


## 2. Run — `obb.backtest.run`

The single-config run. Produces an equity curve, a drawdown series,
per-trade rows, and a summary object (Sharpe, MaxDD, CAGR, turnover,
etc.). This is the "how did it feel" pass — before we ask whether it
was real.

*The code cell below runs the backtest and renders equity curve +
drawdown chart + summary metrics.*

In [ ]:
# [CODE PLACEHOLDER — Phase B] obb.backtest.run(config); render equity curve + drawdown + summary (Sharpe, CAGR, MaxDD, turnover)


## 3. Sweep — `obb.backtest.sweep`

Nothing is more dangerous than a single backtest number. `sweep` runs
the same strategy across a parameter grid (top-K in {3, 5, 7, 10},
rebalance in {weekly, biweekly, monthly, quarterly}) and returns the
whole surface.

The story I care about: is the base-run Sharpe a peak in the middle
of a good neighborhood, or a lonely spike surrounded by rubble? If
neighbor cells are half the Sharpe, I got lucky — I fit a parameter to
history and history won't repeat.

*The code cell below runs the sweep and renders the (top-K,
rebalance-cadence) grid as a Sharpe heatmap.*

In [ ]:
# [CODE PLACEHOLDER — Phase B] obb.backtest.sweep across top-K x rebalance cadence; render Sharpe heatmap; mark the base-run cell


## 4. Validate — walk-forward + PBO

`obb.backtest.validate` does two things:

- **Walk-forward** — retrain on rolling in-sample windows, test on the
  next out-of-sample slice. If in-sample Sharpe is 1.4 and
  out-of-sample is 0.2, the strategy is overfit; the base-run number
  was the in-sample-only view.
- **PBO** — **Probability of Backtest Overfitting**. Under 0.5 means
  "probably real edge"; over 0.5 means "coin flip." This is one of the
  few honest scalar numbers in backtesting.

*The code cell below runs validate and prints the walk-forward Sharpe
distribution + the PBO number.*

In [ ]:
# [CODE PLACEHOLDER — Phase B] obb.backtest.validate with walk-forward + PBO; print WF Sharpe distribution + PBO scalar


## 5. Reading a PBO number honestly

The rule I use:

| PBO | What it means |
|-----|---------------|
| < 0.3 | Reasonable edge; still not a green light, but worth continuing |
| 0.3-0.5 | Borderline; needs longer OOS window before I'd risk capital |
| 0.5-0.7 | Coin flip; the backtest is telling me nothing |
| > 0.7 | Almost certainly overfit; drop the strategy |

The honest thing about this notebook: the toy top-5 owner-earnings
strategy on 10 names is *likely* to score PBO > 0.5. If it does, we
don't rewrite the strategy to get a nicer number. We ship the honest
result and note it. That's the teaching moment — the tool caught what
Sam was about to trade.

## 6. Tearsheet — `obb.backtest.tearsheet`

QuantStats-style HTML tearsheet. Rolling Sharpe, monthly returns
heatmap, drawdown periods, exposure over time. The single-page
dashboard I actually screenshot into my notes.

*The code cell below generates the tearsheet and writes it to
`.notebook_state/tearsheet.html`. Open it in your browser to inspect.*

In [ ]:
# [CODE PLACEHOLDER — Phase B] obb.backtest.tearsheet(config, result); write HTML to .notebook_state/tearsheet.html; print path


## 7. Factor panel + alphalens

Now the mechanical question: **is the strategy's return explained by
known factors, or is there something residual?** `obb.backtest.factor`
regresses returns against a small factor bundle (market, size, value,
momentum, quality). If R² is high, the strategy is just a factor bet
in disguise — cheaper to implement via ETFs.

*The code cell below runs the factor decomposition on the backtest
returns and prints the factor loadings + residual alpha.*

In [ ]:
# [CODE PLACEHOLDER — Phase B] obb.backtest.factor on backtest returns; render factor loadings + residual alpha; comment on whether the strategy is 'just' a factor tilt


## 8. Data bundle — reproducibility

`obb.backtest.bundle_create` freezes the exact input data used for a
backtest — prices, fundamentals, holdings — into a versioned bundle.
Six months from now when I want to know whether the strategy's edge
was real or dead, I re-run against the same bundle and compare.

*The code cell below creates a bundle for this run.*

In [ ]:
# [CODE PLACEHOLDER — Phase B] obb.backtest.bundle_create for this run; print bundle path + hash


## 9. Save state for NB07

`.notebook_state/backtest_result.pkl` + the tearsheet HTML.

*The code cell below pickles the summary result.*

In [ ]:
# [CODE PLACEHOLDER — Phase B] pickle backtest_result to .notebook_state/backtest_result.pkl


---

## What is NOT in this notebook

- **Monte-Carlo bootstrap.** Full bootstrap CI on backtest metrics is future work.
- **Regime-conditional slicing.** The `openbb_regime` extension exists; wiring it into `validate` so we get 'PBO under bear regime vs bull regime' is on the roadmap.
- **Multi-strategy portfolios.** `run` handles one strategy; combining N strategies with risk budgeting is future work.

## Preview of NB07

Everything so far assumed live data. But I don't want a system that only runs when the internet is fast and every provider is up. In NB07 I show how the whole pipeline runs from checked-in snapshots — reproducible on a plane, and again six months from now, from a clean git checkout. And I walk the Monday-morning routine end-to-end: NB01 → NB07 in 45 minutes with one HTML report.
